# Phase 6 v4 — LoRMA32, one model, strict one-hour cap
**Kaggle:** choose **GPU T4 ×2**, enable Internet, attach `phase2v4`, enable secret **HF_WRITE** with write access to `DumbLad/legal-phase6-v4`, then **Save & Run All**.

**No ablation, no baseline-generation run, no student and no knowledge distillation.** This runs one rank-32 LoRMA+ model on the exact Phase 2 v4 base. The original QK-normalized GQA, RoPE and parallel attention/FFN behavior are preserved. Only the original LM/tokenizer source cells are copied; Phase 2 NER heads and its distillation experiments are not imported.

## Complete pipeline
1. Check HF write access before expensive work.
2. Download pinned data/tokenizer revisions and validate the original v4 LM checkpoint.
3. Split documents into disjoint training, loss-validation (64 docs), length-tuning (32 docs), and generation-report (64 docs) groups.
4. Build sentence-aware overlapping windows with up to **896 prompt tokens + 384 target tokens = 1280**. Align reference sentences against all training windows using stemmed lexical similarity, query coverage and exact LCS/ROUGE-L recall. Count imperfect/unmatched supervision.
5. Benchmark generation on both GPUs and reserve the actual complete selected-window inference workload. Source selection retains at least **85% unique source-token coverage per document**, including opening/final windows.
6. Train **one rank-32 identity-inflated LoRMA+ model**, targeting at most two epochs. **The one-hour cap takes precedence:** training stops early if necessary and restores the best trained adapter state by token-weighted validation CE. Actual completed epoch fraction and updates are reported. The notebook never claims two epochs when fewer finished.
7. Generate the selected chunks for all tuning/report/test documents on both GPUs. Deduplicate generated sentences, prefer complete sentences, cover legal roles and use source-support/novelty-aware selection. Tune the final document cap among 640/896/1152 tokens using cached tuning outputs only—no extra training or model-generation runs.
8. Report held-out generation quality, verify complete test predictions, create README/citations/metrics/manifest and upload only the explicit artifact allowlist. Download remote files at the returned commit and compare hashes.

## Strict runtime policy
The global clock starts in settings **before dependency installation**. Every external stage, including HF preflight and final upload, runs through a **59-minute global subprocess deadline**, leaving shutdown overhead before 60 minutes. Training/inference use a 55-minute internal target, with benchmarked inference time reserved first and remaining headroom for packaging/upload. No stage waits out an epoch after the training cutoff.

Two full epochs and a complete upload cannot be guaranteed together within one hour on arbitrary network/hardware. This notebook prioritizes your one-hour requirement. If even the required inference workload cannot fit, it fails explicitly rather than claiming a complete submission with missing records. The real Kaggle runtime and resulting quality remain unverified here.

## Outputs
`/kaggle/working/phase6_v4_submission/submission_lorma/prediction.json` and `submission_lorma.zip` (sole root member: `prediction.json`). The same output root contains a populated README, citations, sanitized config, audit, loss history, actual training/quality metrics, source code and SHA-256 manifest. **No model weights or raw documents/reference summaries are written into the published artifacts.** Best adapters are held in CPU memory; temporary data are deleted after packaging. Upload failure preserves local outputs and can be retried without retraining.


In [ ]:
# Phase 6 v4 settings and full-run clock
import os,sys,json,time,tempfile
from pathlib import Path
RUN_STARTED_AT=time.time()
CONFIG=dict(
    seed=67,checkpoint="/kaggle/input/datasets/koooshiieee/phase2v4/final_model(2).pt",
    dataset_repo="Exploration-Lab/CS-781-Capstone",
    train_remote="Phase_6_Summ/Train/train.json",test_remote="Phase_6_Summ/Test/test.json",
    tokenizer_repo="DumbLad/legal-bpe-tokenizer-phase1-v2",
    runtime=str(Path(tempfile.gettempdir())/"phase6_v4_lorma_runtime"),
    tokenizer_dir=str(Path(tempfile.gettempdir())/"phase6_v4_lorma_runtime"/"tokenizer"),
    submission_dir="/kaggle/working/phase6_v4_submission",
    hf_repo_id="DumbLad/legal-phase6-v4",hf_path_in_repo="",
    epochs=2,rank=32,alpha=32,dropout=.05,
    input_tokens=896,target_tokens=384,chunk_overlap=128,
    inference_coverage=.85,
    alignment_min_score=.12,alignment_min_recall=.12,alignment_sentence_tokens=192,
    holdout_fraction=.05,val_loss_docs=64,val_loss_chunks=256,
    length_tune_docs=32,report_docs=64,
    summary_length_candidates=[640,896,1152],min_chunk_new_tokens=64,max_chunk_new_tokens=384,
    lr=2e-4,weight_decay=.01,warmup_fraction=.02,constant_lr_fraction=.60,min_lr_fraction=.10,
    micro_batch=2,grad_accum=2,eval_batch=2,gen_batch=8,grad_clip=1.,
    eval_every=250,log_every=25,
    max_run_minutes=59,training_and_inference_minutes=55,min_updates=20,started_at=RUN_STARTED_AT,
)
assert CONFIG['epochs']==2 and CONFIG['rank']==32
assert CONFIG['input_tokens']+CONFIG['target_tokens']==1280
assert Path(CONFIG['checkpoint']).is_file(),f"Attach the original phase2v4 LM checkpoint: {CONFIG['checkpoint']}"
Path(CONFIG['runtime']).mkdir(parents=True,exist_ok=True)
Path(CONFIG['submission_dir']).mkdir(parents=True,exist_ok=True)
# Never put HF token values into CONFIG or any saved file.
Path('phase6_v4_config.json').write_text(json.dumps(CONFIG,indent=2),encoding='utf8')


In [ ]:
# Install dependencies, require both GPUs and define the bounded subprocess runner
import subprocess,threading,signal

def run_checked(args):
    remaining=RUN_STARTED_AT+CONFIG['max_run_minutes']*60-time.time()
    assert remaining>0,'Full-run safety deadline reached.'
    process=subprocess.Popen(args,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
    def stream():
        for line in process.stdout:print(line,end='',flush=True)
    thread=threading.Thread(target=stream,daemon=True);thread.start()
    try:status=process.wait(timeout=remaining)
    except subprocess.TimeoutExpired:
        os.killpg(process.pid,signal.SIGTERM)
        try:process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid,signal.SIGKILL);process.wait()
        thread.join(timeout=5)
        raise RuntimeError('Safety deadline reached. An incomplete two-epoch run is not marked successful.')
    thread.join(timeout=10)
    assert status==0,f'Process failed with exit code {status}; inspect the error above.'

run_checked([sys.executable,'-m','pip','install','-q','huggingface_hub','regex','rouge-score','tqdm','--retries','1','--timeout','30'])
import torch
assert torch.cuda.is_available() and torch.cuda.device_count()==2,'Select Kaggle GPU T4 x2.'
print('PyTorch',torch.__version__)
for gpu in range(2):print('GPU',gpu,torch.cuda.get_device_name(gpu))
if not os.environ.get('HF_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_TOKEN']=UserSecretsClient().get_secret('HF')
    except Exception:print('No optional read token; trying public data/tokenizer downloads.')


Exact unchanged source from Phase 2 v4: `%%writefile llama_scratch.py`. Its QK normalization, RoPE, original block structure, GQA projections and tied LM head are preserved.

In [ ]:
%%writefile llama_scratch.py
"""
Optimized Modern Legal LLM (from scratch):
- Pre-norm RMSNorm
- QK-Norm (RMSNorm on Query and Key vectors before attention)
- Rotary Position Embeddings (RoPE)
- Grouped Query Attention (GQA) with fused SDPA
- Parallel Attention & SwiGLU FFN (PaLM / GPT-J style)
- Deep & Narrow parameter presets (MobileLLM efficiency)
- Tied input/output embeddings
"""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F

# ----------------------------------------------------------------------------- Config
@dataclass
class Config:
    vocab_size: int = 33000
    d_model: int = 672
    n_layers: int = 18
    n_heads: int = 14
    n_kv_heads: int = 2          # 14 query heads share 2 KV heads (7:1 GQA)
    ffn_hidden: int = 1792
    max_seq_len: int = 1280
    rope_cache_len: int = 4096
    rope_theta: float = 10000.0
    norm_eps: float = 1e-5
    tie_embeddings: bool = True
    pos_enc: str = "rope"        # "rope"
    attn_type: str = "gqa"       # "gqa"
    attn_impl: str = "sdpa"      # "sdpa" (PyTorch fused kernel)
    use_qk_norm: bool = True     # QK-Norm for training stability
    parallel_attn: bool = True   # Parallel Attention + FFN

PRESETS = {
    "deep_narrow_120m": dict(d_model=640, n_layers=20, n_heads=10, n_kv_heads=2, ffn_hidden=1664),
    "arch_deep_narrow_135m": dict(d_model=672, n_layers=18, n_heads=14, n_kv_heads=2, ffn_hidden=1792),
    "tiny": dict(d_model=64, n_layers=2, n_heads=4, n_kv_heads=2, ffn_hidden=128),
}

def count_params(model, verbose=True) -> int:
    total = sum(p.numel() for p in model.parameters())
    if verbose:
        emb = model.tok_emb.weight.numel()
        blocks = sum(p.numel() for p in model.layers.parameters())
        print(f"Total: {total/1e6:.2f}M | Transformer Blocks: {blocks/1e6:.2f}M | Embeddings: {emb/1e6:.2f}M")
    return total

# ----------------------------------------------------------------------------- RMSNorm
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).type_as(x)

# ----------------------------------------------------------------------------- RoPE
def build_rope_cache(head_dim, max_len, theta):
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t = torch.arange(max_len).float()
    freqs = torch.outer(t, inv_freq)
    return freqs.cos(), freqs.sin()

def apply_rope(x, cos, sin):
    half = x.shape[-1] // 2
    xf = x.float()
    x1, x2 = xf[..., :half], xf[..., half:]
    cos = cos[None, None]
    sin = sin[None, None]
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.type_as(x)

# ----------------------------------------------------------------------------- GQA Attention with QK-Norm
def repeat_kv(x, n_rep):
    if n_rep == 1:
        return x
    B, n_kv, S, hd = x.shape
    return x[:, :, None].expand(B, n_kv, n_rep, S, hd).reshape(B, n_kv * n_rep, S, hd)

class GQAttention(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0 and cfg.n_heads % cfg.n_kv_heads == 0
        self.nh = cfg.n_heads
        self.nkv = cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.n_rep = self.nh // self.nkv
        self.impl = cfg.attn_impl
        self.use_qk_norm = cfg.use_qk_norm

        self.wq = nn.Linear(cfg.d_model, self.nh * self.hd, bias=False)
        self.wk = nn.Linear(cfg.d_model, self.nkv * self.hd, bias=False)
        self.wv = nn.Linear(cfg.d_model, self.nkv * self.hd, bias=False)
        self.wo = nn.Linear(self.nh * self.hd, cfg.d_model, bias=False)

        if self.use_qk_norm:
            self.q_norm = RMSNorm(self.hd, cfg.norm_eps)
            self.k_norm = RMSNorm(self.hd, cfg.norm_eps)

    def forward(self, x, cos, sin, key_valid=None, past=None, use_cache=False):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        k = self.wk(x).view(B, T, self.nkv, self.hd).transpose(1, 2)
        v = self.wv(x).view(B, T, self.nkv, self.hd).transpose(1, 2)

        # Apply QK-Norm
        if self.use_qk_norm:
            q = self.q_norm(q)
            k = self.k_norm(k)

        # Apply RoPE
        if cos is not None:
            q = apply_rope(q, cos, sin)
            k = apply_rope(k, cos, sin)

        if past is not None:
            k = torch.cat([past[0], k], dim=2)
            v = torch.cat([past[1], v], dim=2)
        new_cache = (k, v) if use_cache else None
        S = k.shape[2]

        k = repeat_kv(k, self.n_rep)
        v = repeat_kv(v, self.n_rep)

        if self.impl == "sdpa" and key_valid is None and T == S:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        else:
            qpos = torch.arange(S - T, S, device=x.device)
            kpos = torch.arange(S, device=x.device)
            rel = qpos[:, None] - kpos[None, :]
            allowed = (rel >= 0)[None, None]
            if key_valid is not None:
                allowed = allowed & key_valid[:, None, None, :]
            scores = ((q * (self.hd ** -0.5)) @ k.transpose(-2, -1)).float()
            scores = scores.masked_fill(~allowed, -1e30)
            probs = torch.softmax(scores, dim=-1).to(v.dtype)
            out = probs @ v

        out = out.transpose(1, 2).reshape(B, T, self.nh * self.hd)
        return self.wo(out), new_cache

# ----------------------------------------------------------------------------- SwiGLU FFN
class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden):
        super().__init__()
        self.w_gate = nn.Linear(d_model, hidden, bias=False)
        self.w_up = nn.Linear(d_model, hidden, bias=False)
        self.w_down = nn.Linear(hidden, d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))

# ----------------------------------------------------------------------------- Transformer Block
class Block(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.parallel_attn = cfg.parallel_attn
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = GQAttention(cfg)
        self.mlp_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.mlp = SwiGLU(cfg.d_model, cfg.ffn_hidden)

    def forward(self, x, cos, sin, key_valid=None, past=None, use_cache=False):
        if self.parallel_attn:
            # Parallel Attention & FFN (PaLM style)
            h_attn, new_cache = self.attn(self.attn_norm(x), cos, sin, key_valid, past, use_cache)
            h_mlp = self.mlp(self.mlp_norm(x))
            x = x + h_attn + h_mlp
        else:
            h, new_cache = self.attn(self.attn_norm(x), cos, sin, key_valid, past, use_cache)
            x = x + h
            x = x + self.mlp(self.mlp_norm(x))
        return x, new_cache

# ----------------------------------------------------------------------------- Full LM
class LlamaLM(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.layers = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight

        cos, sin = build_rope_cache(cfg.d_model // cfg.n_heads, cfg.rope_cache_len, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("wo.weight") or name.endswith("w_down.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layers))

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, use_cache=False, cache=None, start_pos=0,
                key_valid=None, all_logits=False):
        B, T = idx.shape
        x = self.tok_emb(idx)
        cos = self.rope_cos[start_pos:start_pos + T]
        sin = self.rope_sin[start_pos:start_pos + T]

        new_caches = []
        for i, layer in enumerate(self.layers):
            past = cache[i] if cache is not None else None
            x, c = layer(x, cos, sin, key_valid=key_valid, past=past, use_cache=use_cache)
            new_caches.append(c)
        x = self.norm(x)

        if targets is not None:
            logits = self.lm_head(x)
            return F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100)
        logits = self.lm_head(x if all_logits else x[:, -1:, :])
        return logits, (new_caches if use_cache else None)

# ----------------------------------------------------------------------------- Generation
@torch.no_grad()
def generate(model, prompts, max_new_tokens, eos_id, pad_id, temperature=0.2, top_k=0,
             top_p=0.9, rep_penalty=1.12, ban_eos=False):
    model.eval()
    device = next(model.parameters()).device
    V = model.cfg.vocab_size
    B = len(prompts)
    L = max(len(p) for p in prompts)
    ids = torch.full((B, L), pad_id, dtype=torch.long, device=device)
    key_valid = torch.zeros(B, L, dtype=torch.bool, device=device)
    seen = torch.zeros(B, V, dtype=torch.bool, device=device)

    for b, p in enumerate(prompts):
        t = torch.tensor(p, dtype=torch.long, device=device)
        ids[b, L - len(p):] = t
        key_valid[b, L - len(p):] = True
        seen[b, t] = True

    out = [[] for _ in range(B)]
    finished = torch.zeros(B, dtype=torch.bool, device=device)
    amp = torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda"))

    with amp:
        logits, cache = model(ids, use_cache=True, key_valid=key_valid, start_pos=0)
        for step in range(max_new_tokens):
            lg = logits[:, -1, :].float()
            if ban_eos:
                lg[:, eos_id] = float("-inf")
            if rep_penalty != 1.0:
                pen = torch.where(lg > 0, lg / rep_penalty, lg * rep_penalty)
                lg = torch.where(seen, pen, lg)

            if temperature <= 0.05:
                nxt = lg.argmax(-1)
            else:
                lg = lg / temperature
                if top_k and top_k > 0:
                    kth = torch.topk(lg, min(top_k, V)).values[:, -1:]
                    lg = lg.masked_fill(lg < kth, float("-inf"))
                if top_p < 1.0:
                    sl, si = torch.sort(lg, descending=True, dim=-1)
                    pr = torch.softmax(sl, dim=-1)
                    drop = (pr.cumsum(-1) - pr) > top_p
                    sl = sl.masked_fill(drop, float("-inf"))
                    lg = torch.full_like(lg, float("-inf")).scatter(1, si, sl)
                nxt = torch.multinomial(torch.softmax(lg, dim=-1), 1).squeeze(1)

            nxt = torch.where(finished, torch.full_like(nxt, pad_id), nxt)
            for b in range(B):
                if not finished[b] and nxt[b].item() != eos_id:
                    out[b].append(int(nxt[b].item()))
            finished = finished | (nxt == eos_id)
            seen.scatter_(1, nxt[:, None], True)
            if bool(finished.all()):
                break
            key_valid = torch.cat([key_valid, torch.ones(B, 1, dtype=torch.bool, device=device)], dim=1)
            logits, cache = model(nxt[:, None], use_cache=True, cache=cache,
                                  start_pos=L + step, key_valid=key_valid)
    return out

Exact unchanged source from Phase 2 v4: `%%writefile bpe_tokenizer.py`. Preparation pins the tokenizer repository revision and records file hashes.

In [ ]:
%%writefile bpe_tokenizer.py
#!/usr/bin/env python3
import os, json, regex
from typing import Dict, List, Tuple
from collections import Counter, defaultdict
from tqdm.auto import tqdm

import regex


_PAT = regex.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")


class PureByteLevelBPETokenizer:
    def __init__(self, vocab_size: int = 33000):
        self.vocab_size = vocab_size
        self.special_tokens = ["<pad>", "<unk>", "<s>", "</s>", "<mask>"]

        self.encoder: Dict[bytes, int] = {}
        self.decoder: Dict[int, bytes] = {}
        self.merges: Dict[Tuple[bytes, bytes], bytes] = {}       # pair -> merged token
        self.merge_ranks: Dict[Tuple[bytes, bytes], int] = {}    # pair -> order learned (lower = earlier)
        self._cache: Dict[str, List[int]] = {}

        self._init_base_vocab()

    def _init_base_vocab(self):
        for b in range(256):
            b_bytes = bytes([b])
            self.encoder[b_bytes] = b
            self.decoder[b] = b_bytes

        # special tokens
        for idx, token in enumerate(self.special_tokens, start=256):
            b_bytes = token.encode("utf-8")
            self.encoder[b_bytes] = idx
            self.decoder[idx] = b_bytes

    @property
    def pad_token_id(self) -> int:
        return self.encoder["<pad>".encode("utf-8")]

    @property
    def unk_token_id(self) -> int:
        return self.encoder["<unk>".encode("utf-8")]

    @property
    def bos_token_id(self) -> int:
        return self.encoder["<s>".encode("utf-8")]

    @property
    def eos_token_id(self) -> int:
        return self.encoder["</s>".encode("utf-8")]

    # ------------------------------------------------------------------ training
    @staticmethod
    def _merge_word(word, a, b, new):
        out, i, n = [], 0, len(word)
        while i < n:
            if i < n - 1 and word[i] == a and word[i + 1] == b:
                out.append(new)
                i += 2
            else:
                out.append(word[i])
                i += 1
        return out

    def train_from_texts(self, texts: List[str], target_vocab_size: int = None, min_word_freq: int = 2):
        if target_vocab_size is None:
            target_vocab_size = self.vocab_size
        num_merges = target_vocab_size - len(self.encoder)
        print(f"\nTraining byte level BPE from scratch. Target merges: {num_merges:,} | Base Vocab: {len(self.encoder)}")

        # 1) pre-tokenize + count unique words (the BPE then runs on this small table, not the raw text)
        word_counts = Counter()
        for text in tqdm(texts, desc="Counting pre-tokens"):
            if text and text.strip():
                word_counts.update(_PAT.findall(text))

        words, freqs = [], []
        for w, c in word_counts.items():
            if c >= min_word_freq:
                words.append([bytes([b]) for b in w.encode("utf-8", "replace")])
                freqs.append(c)
        del word_counts
        print(f"Unique words used for merge learning: {len(words):,}")

        # 2) initial pair statistics (updated incrementally afterwards)
        pair_counts = defaultdict(int)
        where = defaultdict(set)
        for i, w in enumerate(tqdm(words, desc="Initial pair counts")):
            for p in zip(w, w[1:]):
                pair_counts[p] += freqs[i]
                where[p].add(i)

        # 3) merge loop
        pbar = tqdm(total=num_merges, desc="Learning BPE Merges", dynamic_ncols=True)
        for _ in range(num_merges):
            if not pair_counts:
                print("No more candidate pairs found to merge.")
                break

            best_pair, freq = max(pair_counts.items(), key=lambda kv: kv[1])
            if freq < 2:
                print(f"Stopping early: Most frequent pair frequency is {freq}")
                break

            a, b = best_pair
            merged_token = a + b
            self.merges[best_pair] = merged_token
            self.merge_ranks[best_pair] = len(self.merge_ranks)
            if merged_token not in self.encoder:          # two merge paths can give the same bytes
                new_id = len(self.encoder)
                self.encoder[merged_token] = new_id
                self.decoder[new_id] = merged_token

            for i in list(where[best_pair]):
                w, f = words[i], freqs[i]
                for p in zip(w, w[1:]):
                    pair_counts[p] -= f
                    if pair_counts[p] <= 0:
                        del pair_counts[p]
                w = self._merge_word(w, a, b, merged_token)
                words[i] = w
                for p in zip(w, w[1:]):
                    pair_counts[p] += f
                    where[p].add(i)
            del where[best_pair]
            pair_counts.pop(best_pair, None)

            pbar.update(1)
            pbar.set_postfix({"vocab_size": len(self.encoder), "top_pair_freq": freq})

        pbar.close()
        self._cache.clear()
        print(f"Training finished! Final Vocab Size: {len(self.encoder):,}")

    # ------------------------------------------------------------------ encode / decode
    def _encode_word(self, word: str) -> List[int]:
        cached = self._cache.get(word)
        if cached is not None:
            return cached
        tokens = [bytes([b]) for b in word.encode("utf-8", "replace")]
        while len(tokens) >= 2:
            best_pair, best_rank = None, None
            for p in zip(tokens, tokens[1:]):
                r = self.merge_ranks.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best_pair, best_rank = p, r
            if best_pair is None:
                break
            tokens = self._merge_word(tokens, best_pair[0], best_pair[1], self.merges[best_pair])
        ids = [self.encoder.get(t, self.unk_token_id) for t in tokens]
        if len(self._cache) < 1_000_000:
            self._cache[word] = ids
        return ids

    def encode(self, text: str) -> List[int]:
        ids: List[int] = []
        for word in _PAT.findall(text):
            ids.extend(self._encode_word(word))
        return ids

    def decode(self, token_ids: List[int]) -> str:
        raw_bytes = b"".join(self.decoder.get(i, b"<unk>") for i in token_ids)
        return raw_bytes.decode("utf-8", errors="replace")

    # ------------------------------------------------------------------ save / load
    def save(self, save_dir: str):
        os.makedirs(save_dir, exist_ok=True)
        vocab_path = os.path.join(save_dir, "vocab.json")
        merges_path = os.path.join(save_dir, "merges.json")
        config_path = os.path.join(save_dir, "tokenizer_config.json")

        # latin1 maps bytes 1:1 onto code points, so any byte string survives JSON
        ser_vocab = {k.decode("latin1"): v for k, v in self.encoder.items()}
        with open(vocab_path, "w", encoding="utf-8") as f:
            json.dump(ser_vocab, f, ensure_ascii=False, indent=2)

        # JSON (not a space-separated text file): tokens can contain spaces, newlines, \r, \xa0 ...
        ser_merges = [[p1.decode("latin1"), p2.decode("latin1")] for (p1, p2) in self.merges]  # in learned order
        with open(merges_path, "w", encoding="utf-8") as f:
            json.dump(ser_merges, f, ensure_ascii=False)

        config = {
            "tokenizer_class": "PureByteLevelBPETokenizer",
            "vocab_size": len(self.encoder),
            "special_tokens": self.special_tokens,
            "pad_token_id": self.pad_token_id,
            "bos_token_id": self.bos_token_id,
            "eos_token_id": self.eos_token_id
        }
        with open(config_path, "w", encoding="utf-8") as f:
            json.dump(config, f, indent=2)

        print(f"Tokenizer saved successfully to {save_dir}")

    @classmethod
    def load(cls, save_dir: str) -> "PureByteLevelBPETokenizer":
        tok = cls()
        with open(os.path.join(save_dir, "vocab.json"), "r", encoding="utf-8") as f:
            ser_vocab = json.load(f)
        tok.encoder = {k.encode("latin1"): v for k, v in ser_vocab.items()}
        tok.decoder = {v: k.encode("latin1") for k, v in ser_vocab.items()}

        tok.merges, tok.merge_ranks = {}, {}
        with open(os.path.join(save_dir, "merges.json"), "r", encoding="utf-8") as f:
            for rank, (s1, s2) in enumerate(json.load(f)):
                p1, p2 = s1.encode("latin1"), s2.encode("latin1")
                tok.merges[(p1, p2)] = p1 + p2
                tok.merge_ranks[(p1, p2)] = rank
        return tok


Rank-32 LoRMA+, exact v4 forward helpers, faster equivalent masked attention, sentence-aware windows, improved alignment and legal-role-aware generated-sentence reduction.

In [ ]:
%%writefile phase6_v4_common.py
import os, json, math, random, re, time, hashlib
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import asdict
from llama_scratch import Config, LlamaLM
from bpe_tokenizer import PureByteLevelBPETokenizer

def read_config():
    return json.loads(Path('phase6_v4_config.json').read_text())

def seed_all(seed):
    random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def atomic_json(obj, path):
    path=Path(path); tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding='utf8'); tmp.replace(path)

class LoRMALinear(nn.Module):
    # Identity-inflated LoRMA+: W_eff=(I + scale*B@A)@W. A/B act on W*x, not on x.
    def __init__(self,base,rank=16,alpha=16,dropout=.05):
        super().__init__();self.base=base;self.scale=alpha/rank
        self.A=nn.Parameter(torch.empty(rank,base.out_features));self.B=nn.Parameter(torch.zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5));self.drop=nn.Dropout(dropout)
    def forward(self,x):
        y=self.base(x)
        return y+F.linear(F.linear(self.drop(y),self.A),self.B)*self.scale

def inject_lorma(model,c):
    model.requires_grad_(False)
    for block in model.layers:
        for parent,names in [(block.attn,['wq','wk','wv','wo']),(block.mlp,['w_gate','w_up','w_down'])]:
            for name in names:setattr(parent,name,LoRMALinear(getattr(parent,name),c['rank'],c['alpha'],c['dropout']))
    for name,p in model.named_parameters():
        if 'norm' in name:p.requires_grad_(True)
    return model

def merge_lorma(model):
    for parent in list(model.modules()):
        for name,m in list(parent.named_children()):
            if isinstance(m,LoRMALinear):
                with torch.no_grad():m.base.weight.add_((m.B@(m.A@m.base.weight))*m.scale)
                setattr(parent,name,m.base)
    return model

def load_base(c):
    ck=torch.load(c['checkpoint'],map_location='cpu',weights_only=False)
    assert isinstance(ck,dict) and 'cfg' in ck and 'model' in ck,'Use the original v4 LM checkpoint with cfg/model, not NER weights.'
    cfg=Config(**ck['cfg']);cfg.attn_impl='sdpa'
    assert cfg.pos_enc=='rope' and cfg.attn_type=='gqa','Wrong Phase 2 v4 base config'
    assert cfg.d_model//cfg.n_heads % 2==0
    assert c['input_tokens']+c['target_tokens']<=cfg.max_seq_len
    m=LlamaLM(cfg);m.load_state_dict(ck['model'],strict=True)
    assert all(torch.isfinite(p).all() for p in m.parameters()),'Nonfinite input model'
    return m

def hidden(model,x):
    h=model.tok_emb(x);cos=model.rope_cos[:x.shape[1]];sin=model.rope_sin[:x.shape[1]]
    for block in model.layers:h,_=block(h,cos,sin,key_valid=None,past=None,use_cache=False)
    return model.norm(h)

class Objective(nn.Module):
    def __init__(self,model):super().__init__();self.model=model
    def forward(self,x,y):
        h=hidden(self.model,x);mask=y.ne(-100);z=self.model.lm_head(h[mask]).float()
        return F.cross_entropy(z,y[mask])

def collate(examples,pad,device):
    length=max(len(e['x']) for e in examples)
    x=torch.full((len(examples),length),pad,dtype=torch.long); y=torch.full_like(x,-100)
    for i,e in enumerate(examples):
        x[i,:len(e['x'])]=torch.tensor(e['x']); y[i,:len(e['y'])]=torch.tensor(e['y'])
    return x.to(device),y.to(device)

def token_ids(tok,text):
    ids=tok.encode(text); return ids.ids if hasattr(ids,'ids') else ids

def make_target(tok,text,limit):
    raw=token_ids(tok,text)
    # A cropped reference is not an actual end of summary: do not teach false early EOS.
    return (raw+[tok.eos_token_id] if len(raw)<limit else raw[:limit]), len(raw)>=limit

def records(path,training):
    obj=json.loads(Path(path).read_text(encoding='utf8'))
    if isinstance(obj,dict):
        obj=next((obj[k] for k in ['data','records','examples'] if isinstance(obj.get(k),list)),obj)
    assert isinstance(obj,list), 'Expected list of records'
    out=[]; seen=set(); dropped=[]
    for r in obj:
        assert isinstance(r,dict) and 'id' in r and isinstance(r.get('document'),str) and r['document'].strip()
        key=str(r['id']); assert key not in seen, 'Duplicate id'; seen.add(key)
        if training and not (isinstance(r.get('summary'),str) and r['summary'].strip()): dropped.append(key); continue
        out.append(dict(r,id=key))
    return out,dropped

def install_fast_gqa():
    from llama_scratch import GQAttention,apply_rope,repeat_kv
    def forward(self,x,cos,sin,key_valid=None,past=None,use_cache=False):
        B,T,_=x.shape;q=self.wq(x).view(B,T,self.nh,self.hd).transpose(1,2)
        k=self.wk(x).view(B,T,self.nkv,self.hd).transpose(1,2);v=self.wv(x).view(B,T,self.nkv,self.hd).transpose(1,2)
        if self.use_qk_norm:q,k=self.q_norm(q),self.k_norm(k)
        if cos is not None:q,k=apply_rope(q,cos,sin),apply_rope(k,cos,sin)
        if past is not None:k=torch.cat([past[0],k],2);v=torch.cat([past[1],v],2)
        cache=(k,v) if use_cache else None;S=k.shape[2];k=repeat_kv(k,self.n_rep);v=repeat_kv(v,self.n_rep)
        if key_valid is None and T==S:out=F.scaled_dot_product_attention(q,k,v,is_causal=True)
        else:
            rel=torch.arange(S-T,S,device=x.device)[:,None]-torch.arange(S,device=x.device)[None,:]
            allowed=(rel>=0)[None,None]
            if key_valid is not None:allowed=allowed & key_valid[:,None,None,:]
            out=F.scaled_dot_product_attention(q,k,v,attn_mask=allowed)
        return self.wo(out.transpose(1,2).reshape(B,T,self.nh*self.hd)),cache
    GQAttention.forward=forward

@torch.inference_mode()
def decode_batch(model,prompts,tok,max_new,force_length=False):
    model.eval(); dev=next(model.parameters()).device; B=len(prompts); L=max(map(len,prompts))
    assert L+max_new<=min(model.cfg.max_seq_len,model.cfg.rope_cache_len),'Generation would exceed the original context'
    x=torch.full((B,L),tok.pad_token_id,dtype=torch.long,device=dev); valid=torch.zeros_like(x,dtype=torch.bool)
    for i,p in enumerate(prompts): x[i,L-len(p):]=torch.tensor(p,device=dev); valid[i,L-len(p):]=True
    done=torch.zeros(B,dtype=torch.bool,device=dev); out=[[] for _ in prompts]; seen=torch.zeros(B,model.cfg.vocab_size,dtype=torch.bool,device=dev)
    with torch.autocast(dev.type,dtype=torch.float16,enabled=dev.type=='cuda'):
        logits,cache=model(x,key_valid=valid,use_cache=True)
        for j in range(max_new):
            z=logits[:,-1].float(); z[:,[tok.pad_token_id,tok.bos_token_id,tok.unk_token_id]]=-float('inf')
            if j<32 or force_length: z[:,tok.eos_token_id]=-float('inf')
            z=torch.where(seen,torch.where(z>0,z/1.1,z*1.1),z)
            # Ban repeated generated trigrams without penalizing words copied from the source.
            for b,seq in enumerate(out):
                if len(seq)>=2:
                    banned=[seq[k+2] for k in range(len(seq)-2) if seq[k:k+2]==seq[-2:]]
                    if banned: z[b,banned]=-float('inf')
            nxt=z.argmax(-1); nxt=torch.where(done,torch.full_like(nxt,tok.pad_token_id),nxt)
            values=nxt.tolist()
            for b,v in enumerate(values):
                if not done[b].item() and v!=tok.eos_token_id: out[b].append(v)
            done|=nxt.eq(tok.eos_token_id); seen.scatter_(1,nxt[:,None],True)
            if bool(done.all()) or j+1==max_new: break
            valid=torch.cat([valid,torch.ones(B,1,dtype=torch.bool,device=dev)],dim=1)
            logits,cache=model(nxt[:,None],key_valid=valid,use_cache=True,cache=cache,start_pos=L+j)
    return [tok.decode(s).strip() for s in out]

def word_tokens(text):
    from rouge_score.tokenizers import DefaultTokenizer
    if not hasattr(word_tokens,'tokenizer'):word_tokens.tokenizer=DefaultTokenizer(use_stemmer=True)
    return word_tokens.tokenizer.tokenize(text)

def word_features(text):
    stop={'the','and','that','thi','with','from','were','have','been','which','their','there','was','for','are','not','but','had','has','its','court','case'}
    return set(word_tokens(text))-stop

def sentences(text):
    # Approximate legal sentence segmentation; citations/abbreviations remain a known limitation.
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+(?=[A-Z0-9])|\n+',text) if s.strip()]

def lcs_length(a,b):
    # Exact bit-parallel LCS; avoids a quadratic Python DP table during large-scale alignment.
    masks={}
    for i,t in enumerate(b):masks[t]=masks.get(t,0)|(1<<i)
    state=0
    for t in a:
        x=state|masks.get(t,0);state=x & ~(x-((state<<1)|1))
    return state.bit_count()

def role(text):
    low=text.lower()
    if re.search(r'\b(appeal.{0,30}(dismissed|allowed)|we hold|therefore|consequently|petition.{0,25}dismissed|set aside|final order)\b',low):return 'holding'
    if re.search(r'\b(whether|question|issue|dispute concerns|for consideration)\b',low):return 'issue'
    if re.search(r'\b(because|reason|interpret|section|article|evidence|principle|provision|constru|constitution)\b',low):return 'reasoning'
    return 'facts'

def prompt_parts(tok):
    return token_ids(tok,'Legal judgment excerpt:\n'),token_ids(tok,'\nSummarize only the supported facts, issue, reasoning and holding in this excerpt:\n')

def chunk_document(tok,text,c):
    from bpe_tokenizer import _PAT
    import bisect
    prefix,suffix=prompt_parts(tok);budget=c['input_tokens']-1-len(prefix)-len(suffix);overlap=c['chunk_overlap']
    assert 0<overlap<budget and budget>64
    ids=[];boundaries=[]
    for match in _PAT.finditer(text):
        part=match.group();ids.extend(tok._encode_word(part))
        if part.rstrip().endswith(('.', '!', '?')) or '\n' in part:boundaries.append(len(ids))
    assert ids and ids==token_ids(tok,text),'Chunk token stream must exactly match the original tokenizer'
    boundaries=sorted(set(boundaries+[len(ids)]));chunks=[];start=0
    while start<len(ids):
        limit=min(start+budget,len(ids));j=bisect.bisect_right(boundaries,limit)-1
        end=boundaries[j] if j>=0 and boundaries[j]>=start+int(.65*budget) else limit
        if limit==len(ids):end=limit
        body=ids[start:end];decoded=tok.decode(body)
        chunks.append(dict(start=start,end=end,prompt=[tok.bos_token_id]+prefix+body+suffix,text=decoded,role=role(decoded)))
        if end==len(ids):break
        target=end-overlap;j=bisect.bisect_left(boundaries,target)
        candidates=[b for b in boundaries[max(0,j-1):j+2] if start<b<end and abs(b-target)<=64]
        start=min(candidates,key=lambda b:abs(b-target)) if candidates else max(start+1,target)
    assert chunks[0]['start']==0 and chunks[-1]['end']==len(ids)
    assert all(a['start']<b['start']<=a['end'] for a,b in zip(chunks,chunks[1:]))
    return chunks

def interval_coverage(chunks):
    merged=[]
    for ch in sorted(chunks,key=lambda x:x['start']):
        if merged and ch['start']<=merged[-1][1]:merged[-1][1]=max(merged[-1][1],ch['end'])
        else:merged.append([ch['start'],ch['end']])
    return sum(b-a for a,b in merged)

def select_chunks(chunks,fraction):
    assert .5<fraction<=1
    if len(chunks)<=2:return chunks[:]
    n=len(chunks);chosen={0,n-1};priority=[]
    # Role anchors and broad positional coverage, using source text only.
    for label in ['issue','reasoning','facts','holding']:
        eligible=[i for i,ch in enumerate(chunks) if ch['role']==label]
        if eligible:priority.append(eligible[len(eligible)//2])
    queue=[(1,n-2)];spread=[]
    while queue:
        lo,hi=queue.pop(0)
        if lo>hi:continue
        mid=(lo+hi)//2;spread.append(mid);queue.extend([(lo,mid-1),(mid+1,hi)])
    cues={'held','dismiss','allow','appeal','reason','evid','section','constitut'}
    salience={i:len(word_features(chunks[i]['text']) & cues) for i in range(1,n-1)}
    salient=sorted(range(1,n-1),key=salience.__getitem__,reverse=True)
    for a,b in zip(spread,salient):priority.extend([a,b])
    for i in priority:
        if interval_coverage([chunks[j] for j in chosen])/chunks[-1]['end']>=fraction:break
        chosen.add(i)
    result=[chunks[i] for i in sorted(chosen)]
    assert interval_coverage(result)/chunks[-1]['end']>=fraction
    return result

def aligned_examples(tok,record,c):
    from collections import Counter
    chunks=chunk_document(tok,record['document'],c)  # Align against ALL source windows, not a selected subset.
    features=[word_features(ch['text']) for ch in chunks];df=Counter(w for fs in features for w in fs)
    idf={w:1+math.log((len(chunks)+1)/(count+1)) for w,count in df.items()}
    norms=[math.sqrt(sum(idf[w]**2 for w in fs)) or 1 for fs in features]
    source_sentences=[[word_tokens(s)[:c['alignment_sentence_tokens']] for s in sentences(ch['text'])] for ch in chunks]
    assigned=[[] for _ in chunks];omitted=legacy_omitted=0;confidence=[]
    refs=sentences(record['summary'])
    for sentence in refs:
        query=word_features(sentence);qt=word_tokens(sentence)[:c['alignment_sentence_tokens']]
        if not query:omitted+=1;legacy_omitted+=1;continue
        qnorm=math.sqrt(sum(idf.get(w,1)**2 for w in query)) or 1
        cosine=[sum(idf[w]**2 for w in query & fs)/(norm*qnorm) for fs,norm in zip(features,norms)]
        if max(cosine)<.08:legacy_omitted+=1
        shortlist=sorted(range(len(chunks)),key=cosine.__getitem__,reverse=True)[:3];scores=[]
        for i in shortlist:
            recall=sum(idf[w] for w in query & features[i])/max(1e-9,sum(idf.get(w,1) for w in query))
            support=sorted(source_sentences[i],key=lambda ts:len(query & set(ts)),reverse=True)[:2]
            rouge_recall=max((lcs_length(qt,ts)/max(1,len(qt)) for ts in support),default=0)
            combined=.35*cosine[i]+.35*recall+.30*rouge_recall
            scores.append((combined,recall,i))
        score,recall,best=max(scores)
        if score>=c['alignment_min_score'] and recall>=c['alignment_min_recall']:
            assigned[best].append(sentence);confidence.append(score)
        else:omitted+=1
    examples=[]
    for i,(ch,gold) in enumerate(zip(chunks,assigned)):
        if not gold:continue
        target,truncated=make_target(tok,' '.join(gold),c['target_tokens']);prompt=ch['prompt']
        examples.append(dict(id=record['id'],chunk_index=i,x=(prompt+target)[:-1],y=[-100]*(len(prompt)-1)+target,truncated=truncated))
    return examples,dict(source_chunks=len(chunks),reference_sentences=len(refs),unaligned_reference_sentences=omitted,legacy_cosine_only_unaligned=legacy_omitted,mean_alignment_confidence=sum(confidence)/max(1,len(confidence)))

def prediction_tasks(rows,c):
    tasks=[]
    for index,row in enumerate(rows):
        count=len(row['chunks']);cap=max(c['min_chunk_new_tokens'],min(c['max_chunk_new_tokens'],math.ceil(max(c['summary_length_candidates'])/count)))
        cap=int(math.ceil(cap/16)*16)
        for ci,ch in enumerate(row['chunks']):tasks.append(dict(index=index,chunk_index=ci,prompt=ch['prompt'],max_new=cap))
    return tasks

def task_batches(tasks,c,rank):
    ordered=sorted(tasks,key=lambda t:(t['max_new'],len(t['prompt']),t['index'],t['chunk_index']))
    mine=ordered[rank::2];batches=[]
    for task in mine:
        if not batches or len(batches[-1])==c['gen_batch'] or batches[-1][0]['max_new']!=task['max_new']:batches.append([])
        batches[-1].append(task)
    return batches

def candidate_sentences(tok,texts,chunks):
    candidates=[]
    for ci,text in enumerate(texts):
        parts=sentences(text)
        # Prefer complete sentences; discard a trailing cut-off fragment if complete sentences exist.
        complete=[s for s in parts if re.search(r'[.!?][\"\')\]]*$',s)]
        if complete:parts=complete
        source=word_features(chunks[ci]['text'])
        for sentence in parts:
            fs=word_features(sentence)
            if not fs:continue
            if any(len(fs & p['features'])/max(1,len(fs | p['features']))>=.78 for p in candidates):continue
            support=len(fs & source)/max(1,len(fs))
            candidates.append(dict(chunk_index=ci,order=len(candidates),text=sentence,features=fs,role=role(sentence),support=support,tokens=len(token_ids(tok,sentence+' '))))
    assert candidates,'No usable generated sentences; no extractive source fallback is substituted'
    return candidates

def reduce_summary(tok,texts,chunks,budget):
    pool=candidate_sentences(tok,texts,chunks);chosen=[];remaining=budget;roles=set()
    def add(i):
        nonlocal remaining
        if i not in chosen and pool[i]['tokens']<=remaining:chosen.append(i);remaining-=pool[i]['tokens'];roles.add(pool[i]['role']);return True
        return False
    def relevance(p):
        return .55*p['support']+.15*min(1,len(p['features'])/25)+(.15 if p['role']=='reasoning' else 0)+(.15 if p['role']=='holding' and p['chunk_index']==len(chunks)-1 else 0)
    # Cover the available legal roles before the novelty-aware fill; labels are heuristics.
    for label in ['holding','issue','reasoning','facts']:
        eligible=[i for i,p in enumerate(pool) if p['role']==label]
        for i in sorted(eligible,key=lambda i:relevance(pool[i]),reverse=True):
            if add(i):break
    while True:
        eligible=[i for i,p in enumerate(pool) if i not in chosen and p['tokens']<=remaining]
        if not eligible:break
        def mmr(i):
            p=pool[i];redundancy=max((len(p['features'] & pool[j]['features'])/max(1,len(p['features'] | pool[j]['features'])) for j in chosen),default=0)
            role_bonus=.12 if p['role'] not in roles else 0
            return .75*relevance(p)-.25*redundancy+role_bonus
        add(max(eligible,key=mmr))
    if not chosen:
        # Generated text only, limited to the requested token budget.
        return tok.decode(token_ids(tok,pool[0]['text'])[:budget]).strip()
    result=' '.join(pool[i]['text'] for i in sorted(chosen))
    while len(token_ids(tok,result))>budget and len(chosen)>1:
        chosen.pop();result=' '.join(pool[i]['text'] for i in sorted(chosen))
    assert result.strip() and len(token_ids(tok,result))<=budget
    return result.strip()

def chunk_map(rows,shards):
    mapping={}
    for p in shards:
        key=(p['index'],p['chunk_index']);assert key not in mapping,'Duplicate chunk prediction';assert p['summary'].strip()
        mapping[key]=p['summary']
    expected={(i,ci) for i,r in enumerate(rows) for ci in range(len(r['chunks']))}
    assert set(mapping)==expected,'Missing or unexpected selected chunk prediction'
    return mapping

def assemble_predictions(tok,rows,shards,budget):
    mapping=chunk_map(rows,shards)
    return [dict(index=i,id=r['id'],split=r['split'],summary=reduce_summary(tok,[mapping[i,ci] for ci in range(len(r['chunks']))],r['chunks'],budget)) for i,r in enumerate(rows)]


Preparation with disjoint fixed holdouts, genuine-checkpoint checks, benchmarked inference reservation, time-budgeted single-model training and complete selected-window generation.

In [ ]:
%%writefile phase6_v4_engine.py
import pickle,contextlib,gc,shutil
from phase6_v4_common import *

def dump_pickle(obj,path):
    path=Path(path);tmp=path.with_suffix('.tmp')
    with tmp.open('wb') as f:pickle.dump(obj,f,protocol=pickle.HIGHEST_PROTOCOL)
    tmp.replace(path)

def load_pickle(path):
    with Path(path).open('rb') as f:return pickle.load(f)

def sha256(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''):digest.update(block)
    return digest.hexdigest()

def stratified_sample(rows,n,seed):
    assert len(rows)>=n and n>0
    ordered=sorted(rows,key=lambda r:(len(r['document']),r['id']));rng=random.Random(seed)
    return [ordered[rng.randrange(i*len(rows)//n,(i+1)*len(rows)//n)] for i in range(n)]

def balanced_examples(groups,limit,seed):
    rng=random.Random(seed);groups=[g[:] for g in groups if g]
    for group in groups:rng.shuffle(group)
    result=[];depth=0
    while len(result)<limit:
        added=False
        for group in groups:
            if depth<len(group):result.append(group[depth]);added=True
            if len(result)==limit:break
        if not added:break
        depth+=1
    return result

def prepare(c):
    from huggingface_hub import hf_hub_download,HfApi
    runtime=Path(c['runtime']);out=Path(c['submission_dir']);runtime.mkdir(parents=True,exist_ok=True);out.mkdir(parents=True,exist_ok=True)
    token=os.environ.get('HF_TOKEN') or os.environ.get('HF_WRITE_TOKEN')
    api=HfApi(token=token)
    dataset_revision=api.dataset_info(c['dataset_repo']).sha;tokenizer_revision=api.model_info(c['tokenizer_repo']).sha
    paths=[hf_hub_download(c['dataset_repo'],remote,repo_type='dataset',token=token,revision=dataset_revision) for remote in [c['train_remote'],c['test_remote']]]
    for name in ['vocab.json','merges.json','tokenizer_config.json']:hf_hub_download(c['tokenizer_repo'],name,local_dir=c['tokenizer_dir'],token=token,revision=tokenizer_revision)
    tok=PureByteLevelBPETokenizer.load(c['tokenizer_dir']);base=load_base(c)
    assert tok.vocab_size==base.cfg.vocab_size
    assert [tok.pad_token_id,tok.unk_token_id,tok.bos_token_id,tok.eos_token_id]==[256,257,258,259]
    train,dropped=records(paths[0],True);test,_=records(paths[1],False)
    rng=random.Random(c['seed']);rng.shuffle(train)
    needed=c['val_loss_docs']+c['length_tune_docs']+c['report_docs'];nh=max(needed,round(len(train)*c['holdout_fraction']))
    assert nh<len(train) and test,'Dataset too small for the configured disjoint holdout groups'
    holdout=train[:nh];train=train[nh:]
    lossdocs=stratified_sample(holdout,c['val_loss_docs'],c['seed']+1);used={r['id'] for r in lossdocs}
    tunedocs=stratified_sample([r for r in holdout if r['id'] not in used],c['length_tune_docs'],c['seed']+2);used|={r['id'] for r in tunedocs}
    reportdocs=stratified_sample([r for r in holdout if r['id'] not in used],c['report_docs'],c['seed']+3)
    assert not ({r['id'] for r in reportdocs}&used)
    examples=[];audits=[];empty=[]
    for i,r in enumerate(train):
        ex,audit=aligned_examples(tok,r,c);examples.extend(ex);audits.append(audit)
        if not ex:empty.append(r['id'])
        if (i+1)%250==0:print(f'Aligned {i+1}/{len(train)} training documents; {len(examples)} examples',flush=True)
    valg=[]
    for r in lossdocs:valg.append(aligned_examples(tok,r,c)[0])
    val=balanced_examples(valg,c['val_loss_chunks'],c['seed']);assert len(examples)>=64 and val
    assert len({e['id'] for e in val})>=min(c['val_loss_docs']//2,len(lossdocs)),'Too few loss-validation documents aligned'
    inputs=[];coverages=[]
    for split,rows in [('length_tune',tunedocs),('report',reportdocs),('test',test)]:
        for r in rows:
            allchunks=chunk_document(tok,r['document'],c);selected=select_chunks(allchunks,c['inference_coverage'])
            coverages.append(interval_coverage(selected)/allchunks[-1]['end'])
            inputs.append(dict(id=r['id'],split=split,chunks=selected,reference=r.get('summary')))
    # Original next-token LM sanity is distinct from summarization loss.
    base=base.to('cuda:0').eval();sanities=[]
    with torch.no_grad(),torch.autocast('cuda',dtype=torch.float16):
        for r in train[:4]:
            ids=token_ids(tok,r['document'])[:256]
            if len(ids)>1:sanities.append(float(base(torch.tensor([ids[:-1]],device='cuda:0'),targets=torch.tensor([ids[1:]],device='cuda:0'))))
    sanity=sum(sanities)/len(sanities);assert math.isfinite(sanity) and sanity<8,'Bad LM sanity: verify original v4 LM weights and tokenizer'
    dump_pickle(dict(train=examples,val=val,inputs=inputs),runtime/'prepared.pkl')
    report=dict(dataset_revision=dataset_revision,tokenizer_revision=tokenizer_revision,base_cfg=asdict(base.cfg),base_sha256=sha256(c['checkpoint']),tokenizer_sha256={n:sha256(Path(c['tokenizer_dir'])/n) for n in ['vocab.json','merges.json','tokenizer_config.json']},lm_sanity_loss=sanity,training_documents=len(train),training_examples=len(examples),documents_without_aligned_targets=empty,loss_validation_documents=len({e['id'] for e in val}),loss_validation_examples=len(val),length_tuning_documents=len(tunedocs),report_documents=len(reportdocs),test_documents=len(test),test_selected_chunks=sum(len(r['chunks']) for r in inputs if r['split']=='test'),dropped_empty_summary_ids=dropped,target_truncation_fraction=sum(e['truncated'] for e in examples)/len(examples),alignment=dict(method='stemmed source-local IDF cosine + weighted query recall + exact LCS/ROUGE-L recall on supporting source sentences; shortlist top 3 windows',reference_sentences=sum(a['reference_sentences'] for a in audits),unaligned_reference_sentences=sum(a['unaligned_reference_sentences'] for a in audits),legacy_cosine_only_unaligned=sum(a['legacy_cosine_only_unaligned'] for a in audits)),inference_source_coverage=dict(required=c['inference_coverage'],minimum=min(coverages),mean=sum(coverages)/len(coverages)),splits=dict(train=[r['id'] for r in train],loss_validation=[r['id'] for r in lossdocs],length_tune=[r['id'] for r in tunedocs],report=[r['id'] for r in reportdocs]))
    atomic_json(report,out/'data_audit.json');print(json.dumps({k:v for k,v in report.items() if k not in ['splits','documents_without_aligned_targets']},indent=2),flush=True)

def eval_loss(obj,examples,c,device,rank):
    import torch.distributed as dist
    obj.eval();sums=torch.zeros(2,device=device,dtype=torch.float64)
    mine=examples[rank::2]
    with torch.no_grad():
        for at in range(0,len(mine),c['eval_batch']):
            x,y=collate(mine[at:at+c['eval_batch']],c['pad'],device)
            with torch.autocast('cuda',dtype=torch.float16):ce=obj(x,y)
            tokens=y.ne(-100).sum();sums[0]+=ce.double()*tokens;sums[1]+=tokens
    dist.all_reduce(sums);value=float(sums[0]/sums[1]);assert math.isfinite(value)
    return value

def generate_shard(model,rows,tok,c,rank,label):
    result=[];batches=task_batches(prediction_tasks(rows,c),c,rank)
    for at,batch in enumerate(batches):
        assert time.time()<c['started_at']+c['max_run_minutes']*60-60,'Safety deadline reached before complete inference; no full run is claimed'
        texts=decode_batch(model,[t['prompt'] for t in batch],tok,batch[0]['max_new'])
        for task,text in zip(batch,texts):
            assert text.strip(),f'Empty model generation for document {rows[task["index"]]["id"]}'
            result.append(dict(index=task['index'],chunk_index=task['chunk_index'],summary=text))
        atomic_json(result,Path(c['runtime'])/f'{label}_rank{rank}.json')
        if at%10==0 or at+1==len(batches):print(f'{label} GPU{rank}: {at+1}/{len(batches)} chunk batches',flush=True)
    return result

def learning_rate(step,total,c):
    warmup=max(1,min(total,int(total*c['warmup_fraction'])))
    if step<=warmup:return c['lr']*step/warmup
    progress=step/total
    if progress<=c['constant_lr_fraction']:return c['lr']
    phase=min(1,(progress-c['constant_lr_fraction'])/(1-c['constant_lr_fraction']))
    return c['lr']*(c['min_lr_fraction']+(1-c['min_lr_fraction'])*.5*(1+math.cos(math.pi*phase)))

def epoch_order(n,global_batch,seed,epoch):
    order=list(range(n));random.Random(seed+epoch).shuffle(order)
    padding=(-n)%global_batch
    order+=order[:padding]
    return order,padding

def distributed_run(c):
    import torch.distributed as dist
    from torch.nn.parallel import DistributedDataParallel as DDP
    from datetime import timedelta
    rank=int(os.environ['LOCAL_RANK']);torch.cuda.set_device(rank);device=torch.device('cuda',rank)
    dist.init_process_group('nccl',timeout=timedelta(minutes=30));assert dist.get_world_size()==2,'Choose GPU T4 x2'
    seed_all(c['seed']);torch.set_num_threads(2);install_fast_gqa()
    tok=PureByteLevelBPETokenizer.load(c['tokenizer_dir']);c['pad']=tok.pad_token_id
    data=load_pickle(Path(c['runtime'])/'prepared.pkl');model=load_base(c).to(device).eval()
    # One model only: reserve the complete inference workload before allocating training time.
    prompts=sorted([ch['prompt'] for row in data['inputs'] for ch in row['chunks']],key=len,reverse=True)[:c['gen_batch']]
    torch.cuda.synchronize();benchmark_start=time.time();decode_batch(model,prompts,tok,64,force_length=True);torch.cuda.synchronize()
    benchmark=torch.tensor(time.time()-benchmark_start,device=device);dist.all_reduce(benchmark,op=dist.ReduceOp.MAX)
    tasks=prediction_tasks(data['inputs'],c)
    work=max(sum(batch[0]['max_new']/64 for batch in task_batches(tasks,c,r)) for r in [0,1])
    reserve=max(8*60,float(benchmark)*work*1.5+120)
    training_deadline=c['started_at']+c['training_and_inference_minutes']*60-reserve
    allowed=torch.tensor(training_deadline-time.time(),device=device);dist.all_reduce(allowed,op=dist.ReduceOp.MIN)
    assert float(allowed)>120,'Complete prediction workload leaves less than two minutes for training within one hour. No incomplete submission is claimed.'
    if rank==0:print(f'One-model generation benchmark={float(benchmark):.2f}s/64-token batch; inference reserve={reserve/60:.1f}m; training allowance~{float(allowed)/60:.1f}m',flush=True)
    dist.barrier()
    inject_lorma(model,c);model=model.to(device);obj=Objective(model);ddp=DDP(obj,device_ids=[rank],broadcast_buffers=False)
    params=[p for p in model.parameters() if p.requires_grad];names={n for n,p in model.named_parameters() if p.requires_grad}
    bs=c['micro_batch'];accum=c['grad_accum'];global_batch=2*bs*accum
    steps_per_epoch=math.ceil(len(data['train'])/global_batch);total=steps_per_epoch*c['epochs']
    opt=torch.optim.AdamW(params,lr=learning_rate(1,total,c),weight_decay=c['weight_decay']);scaler=torch.amp.GradScaler('cuda',init_scale=1024)
    baseline=eval_loss(obj,data['val'],c,device,rank);best=float('inf');best_state=None;history=[];best_step=0;successful=attempted=skipped=0;start=time.time()
    probe=next(p for n,p in model.named_parameters() if n.endswith('.B'));initial_probe=probe.detach().clone()
    if rank==0:print(f'TRAINING TARGET (one-hour cap takes precedence): {c["epochs"]} epochs x {steps_per_epoch} updates = {total}; global batch={global_batch}; trainable={sum(p.numel() for p in params):,}; baseline validation CE={baseline:.4f}',flush=True)
    def evaluate(epoch):
        nonlocal best,best_state,best_step
        value=eval_loss(obj,data['val'],c,device,rank)
        if value<best:
            best=value;best_step=successful;best_state={n:p.detach().cpu().clone() for n,p in model.named_parameters() if n in names}
        row=dict(epoch=epoch,successful_updates=successful,attempted_updates=attempted,amp_skips=skipped,validation_ce=value,best_validation_ce=best,lr=opt.param_groups[0]['lr'],seconds=time.time()-start)
        history.append(row)
        if rank==0:atomic_json(history,Path(c['submission_dir'])/'training_history.json');print(row,flush=True)
    padding_counts=[];stop_for_time=False;epochs_completed=0;fractional_epochs=0.
    for epoch in range(c['epochs']):
        order,padding=epoch_order(len(data['train']),global_batch,c['seed'],epoch);padding_counts.append(padding)
        for at in range(0,len(order),global_batch):
            time_left=torch.tensor(training_deadline-time.time(),device=device);dist.all_reduce(time_left,op=dist.ReduceOp.MIN)
            if float(time_left)<45:
                stop_for_time=True;break
            ids=order[at:at+global_batch];global_tokens=sum(sum(t!=-100 for t in data['train'][i]['y']) for i in ids)
            for retry in range(6):
                ddp.train();opt.zero_grad(set_to_none=True);local_sum=local_tokens=0.
                for micro in range(accum):
                    offset=micro*2*bs+rank*bs;examples=[data['train'][i] for i in ids[offset:offset+bs]]
                    x,y=collate(examples,c['pad'],device)
                    with (ddp.no_sync() if micro+1<accum else contextlib.nullcontext()):
                        with torch.autocast('cuda',dtype=torch.float16):ce=ddp(x,y);loss=ce*y.ne(-100).sum()*2/global_tokens
                        finite=torch.tensor(int(torch.isfinite(loss)),device=device);dist.all_reduce(finite,op=dist.ReduceOp.MIN)
                        assert finite.item(),'Nonfinite loss, stopped before a corrupt update'
                        scaler.scale(loss).backward()
                    nt=int(y.ne(-100).sum());local_sum+=float(ce.detach())*nt;local_tokens+=nt
                attempted+=1;scaler.unscale_(opt);norm=torch.nn.utils.clip_grad_norm_(params,c['grad_clip'])
                old=scaler.get_scale();scaler.step(opt);scaler.update();did=int(scaler.get_scale()>=old)
                flags=torch.tensor([did,did],device=device);dist.all_reduce(flags[:1],op=dist.ReduceOp.MIN);dist.all_reduce(flags[1:],op=dist.ReduceOp.MAX);assert flags[0]==flags[1]
                if did:break
                skipped+=1
            assert did,'Six consecutive AMP-overflow retries; stopped without skipping this training batch'
            successful+=1;fractional_epochs=epoch+(at+global_batch)/len(order)
            if successful==1:assert float((probe.detach()-initial_probe).abs().max())>0,'Adapter weights did not update'
            if rank==0 and (successful%c['log_every']==0 or successful==1):
                speed=(time.time()-start)/successful;eta=speed*(total-successful)/60
                print(f'epoch={epoch+1}/{c["epochs"]} updates={successful}/{total} local_token_CE={local_sum/max(1,local_tokens):.4f} LR={opt.param_groups[0]["lr"]:.3g} grad_before_clip={float(norm):.3f} AMP_skips={skipped} ETA_to_two_epoch_target~{eta:.1f}m train_budget_left~{max(0,(training_deadline-time.time())/60):.1f}m',flush=True)
            if successful%c['eval_every']==0:evaluate(epoch+1)
            for group in opt.param_groups:group['lr']=learning_rate(min(total,successful+1),total,c)
        if successful and (not history or history[-1]['successful_updates']!=successful):evaluate(epoch+1)
        if not stop_for_time:epochs_completed=epoch+1
        dist.barrier()
        if stop_for_time:break
    assert successful>=c['min_updates'] and best_state is not None,'Insufficient successful updates for a valid model'
    if rank==0 and stop_for_time:print(f'ONE-HOUR BUDGET: stopped at {fractional_epochs:.3f} epochs; restoring the best trained state, then generating every prediction.',flush=True)
    result=model.load_state_dict(best_state,strict=False);assert not result.unexpected_keys and not (set(result.missing_keys)&names)
    if rank==0:
        summary=dict(epochs_completed=epochs_completed,fractional_epochs_completed=fractional_epochs,stopped_for_time=stop_for_time,examples_per_epoch=len(data['train']),steps_per_epoch=steps_per_epoch,successful_updates=successful,expected_updates=total,padded_examples_per_epoch=padding_counts,attempted_updates=attempted,amp_skips=skipped,baseline_validation_ce=baseline,best_validation_ce=best,best_step=best_step,best_fractional_epoch=best_step/steps_per_epoch,monitored_weight_change=float((probe.detach()-initial_probe).abs().max()),trainable_parameters=sum(p.numel() for p in params),training_seconds=time.time()-start,gpus=[torch.cuda.get_device_name(i) for i in range(2)],torch_version=torch.__version__)
        atomic_json(summary,Path(c['submission_dir'])/'training_metrics.json');print('SELECTED BEST BY VALIDATION LOSS:',summary,flush=True)
    del ddp,obj,opt,scaler,best_state;gc.collect();torch.cuda.empty_cache();merge_lorma(model);model.eval()
    generate_shard(model,data['inputs'],tok,c,rank,'lorma');dist.barrier();dist.destroy_process_group()

if __name__=='__main__':
    import sys
    c=read_config();seed_all(c['seed'])
    {'prepare':prepare,'distributed':distributed_run}[sys.argv[1]](c)


Tune the final summary cap, report base/model metrics on 64 independent documents, verify predictions, generate the README and artifact manifest, and clean temporary data.

In [ ]:
%%writefile phase6_v4_report.py
from phase6_v4_common import *
from phase6_v4_engine import load_pickle,sha256
import zipfile,shutil,tempfile

def scores(rows,predictions):
    from rouge_score import rouge_scorer
    assert len(rows)==len(predictions) and rows
    scorer=rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'],use_stemmer=True)
    pairs=[scorer.score(r['reference'],p['summary']) for r,p in zip(rows,predictions)]
    result={key:sum(s[key].fmeasure for s in pairs)/len(pairs) for key in ['rouge1','rouge2','rougeL']}
    result.update(documents=len(rows),mean_prediction_words=sum(len(p['summary'].split()) for p in predictions)/len(rows),mean_reference_words=sum(len(r['reference'].split()) for r in rows)/len(rows))
    return result

def read_shards(runtime,label):
    result=[]
    for rank in [0,1]:result+=json.loads((runtime/f'{label}_rank{rank}.json').read_text(encoding='utf8'))
    return result

def subset_shards(rows,shards,split):
    selected=[i for i,r in enumerate(rows) if r['split']==split];remap={old:new for new,old in enumerate(selected)}
    return [rows[i] for i in selected],[dict(p,index=remap[p['index']]) for p in shards if p['index'] in remap]

def readme(c,audit,training,metrics):
    table='\n'.join(f'| {label} | {values["rouge1"]:.4f} | {values["rouge2"]:.4f} | {values["rougeL"]:.4f} | {values["mean_prediction_words"]:.1f} |' for label,values in metrics['report'].items())
    return f'''---
language:
- en
tags:
- legal
- summarization
- lorma
- predictions-only
datasets:
- {c['dataset_repo']}
---

# Phase 6 v4: legal summarization with rank-32 LoRMA+

This repository contains **prediction artifacts and reproducibility metadata**, not downloadable model weights or a hosted inference model. Results below are generated by the completed run; they are not synthetic authoring-test scores.

## Base and provenance

The exact architecture and tokenizer source are copied from the user's `phase-2-v4 (7).ipynb`. The original LM checkpoint is `{c['checkpoint']}` (SHA-256: `{audit['base_sha256']}`). Its actual config is recorded in `data_audit.json`. The model retains QK-normalized GQA, RoPE, pre-norm RMSNorm, the original parallel-attention/FFN setting, SwiGLU and tied embeddings. Its layers are not pruned or replaced with the v2/v3 architecture.

Tokenizer: [{c['tokenizer_repo']}](https://huggingface.co/{c['tokenizer_repo']}); exact input file hashes are in `data_audit.json`. Data: [{c['dataset_repo']}](https://huggingface.co/datasets/{c['dataset_repo']}), `{c['train_remote']}` and `{c['test_remote']}`. Pinned dataset revision: `{audit.get('dataset_revision','not recorded')}`. Pinned tokenizer revision: `{audit.get('tokenizer_revision','not recorded')}`. Data and base weights are not redistributed here, and no new license for them is asserted.

## Complete pipeline

1. Remove empty-summary training records and create a fixed document-level holdout.
2. Create disjoint, length-stratified holdout groups: {audit['loss_validation_documents']} documents represented in loss evaluation, {audit['length_tuning_documents']} documents for output-length tuning, and {audit['report_documents']} documents for the generation report. Reporting documents never participate in gradient updates, loss-checkpoint selection or output-length tuning.
3. Create sentence-aware overlapping windows with up to {c['input_tokens']} prompt tokens and approximately {c['chunk_overlap']} source tokens of overlap. Oversized sentences fall back to token windows. Every source token is represented in the initial window stream.
4. Align reference sentences against **all training source windows** with stemmed source-local IDF cosine, query recall and exact LCS/ROUGE-L recall over supporting sentences. Reference text is used only to construct training/loss-validation targets. Poor matches remain omitted and are counted; improved alignment is not assumed to be perfect.
5. Train rank-{c['rank']} identity-inflated **LoRMA+**, alpha {c['alpha']}, dropout {c['dropout']}, plus normalization weights. The formula is `W_eff = (I + (alpha/r) B A) W`; this is not additive LoRA or permutation-inflation LoRMAπ. Linear base weights and tied embeddings are frozen.
6. Aim for {c['epochs']} epochs; the one-hour cap takes precedence. This run completed {training['fractional_epochs_completed']:.3f} epochs over {training['examples_per_epoch']} aligned examples per epoch. Distributed batches may repeat at most one global batch's worth of padding examples per epoch; exact counts are in `training_metrics.json`. Select the trained adapter state with the **lowest token-weighted validation CE**. The runtime cap may stop training early; actual progress is reported without claiming two completed epochs.
7. Use a {c['warmup_fraction']:.0%} linear LR warmup, hold the base LR until {c['constant_lr_fraction']:.0%} of training, then cosine-decay smoothly to {c['min_lr_fraction']:.0%} of the base LR at the final update. Train and generate on both GPUs.
8. Select source windows by legal-role anchors and broad positional coverage; actual per-document unique source-token coverage is at least {c['inference_coverage']:.0%}. Every test document receives a prediction. This is source-only selection, not external RAG or reference-based test retrieval.
9. Generate chunk summaries with a latent-free ordinary GQA KV cache. Fuse masked decoding through SDPA while preserving QK normalization and RoPE. Restore the best adapters and merge them in GPU memory; no weight checkpoint is written.
10. Remove near-duplicate generated sentences, prefer complete sentences and source-supported content, cover available facts/issue/reasoning/holding roles, and fill using a novelty-aware MMR-style rule. Restore selected sentences to document order. The reducer is a **custom deterministic heuristic**, not a separately trained model, a reproduction of SUMM^N, or an implementation of Elaraby et al.'s system.
11. Choose one final-summary cap from {c['summary_length_candidates']} tokens using ROUGE-L on the length-tuning group only. The selected cap is **{metrics['selected_summary_tokens']} tokens**. Apply it to the separate report group and every test document.
12. Verify complete selected-window and test-ID coverage, package `prediction.json`, and upload only an explicit artifact allowlist. Remote files are downloaded at the returned commit and checked by SHA-256 before upload success is reported.

## Training result

- Fully completed epochs: **{training['epochs_completed']}**; total fractional progress **{training['fractional_epochs_completed']:.3f}**. Stopped for time: **{training['stopped_for_time']}**.
- Successful optimizer updates: **{training['successful_updates']} / {training['expected_updates']}**.
- Baseline validation CE: **{training['baseline_validation_ce']:.4f}**.
- Best validation CE: **{training['best_validation_ce']:.4f}**, update **{training['best_step']}** (epoch fraction {training['best_fractional_epoch']:.3f}).
- AMP-skipped attempts: {training['amp_skips']}; overflowing batches are retried rather than silently skipped.
- Trainable parameters: {training['trainable_parameters']:,}.

## Independent generation report

The {c['report_docs']}-document report group is disjoint from gradient training, loss-based checkpoint selection and length tuning. Only the fine-tuned LoRMA model is run: no ablation, baseline-generation experiment, student or distillation stage. These are held-out **validation** results, not official test scores. The report is meaningful only if future changes are not repeatedly tuned against it; use a fresh audit split for extensive repeated experiments.

| Model | ROUGE-1 F1 | ROUGE-2 F1 | ROUGE-L F1 | Mean prediction words |
|---|---:|---:|---:|---:|
{table}

ROUGE uses stemming via `rouge-score`. Source-copy rate, factuality and legal correctness are not established by ROUGE. Output cap tuning results, length statistics and all reported scores are in `metrics.json`. No BERTScore or third-party pretrained scorer is silently downloaded.

## Submission layout

```text
submission_lorma/
  prediction.json
submission_lorma.zip   # exactly one root member: prediction.json
```

The JSON is a list of objects with string `id` and nonempty string `summary`, preserving the input test order. There are {audit['test_documents']} test predictions. README, citations, sanitized config, data audit, loss history, training metrics, generation metrics, source code and a SHA-256 manifest are supplementary artifacts. No raw documents, reference summaries, credentials, runtime pickle files or model weights are uploaded.

## Limits and runtime

Target truncation fraction: {audit['target_truncation_fraction']:.2%}. Unaligned reference sentences: {audit['alignment']['unaligned_reference_sentences']} / {audit['alignment']['reference_sentences']}. Clause segmentation, alignment, role labels, source support and redundancy checks are heuristic. Neither partial source coverage nor final-length compression guarantees retention of every legal fact. Summaries may hallucinate, omit, or conflate facts and require substantive evaluation before reliance.

This is one time-budgeted LoRMA run. Training reserves benchmarked inference time and may stop before two epochs. A {c['max_run_minutes']}-minute global subprocess deadline includes dependency installation, preparation, training, generation, packaging and upload. Complete prediction artifacts are never fabricated if the workload cannot fit. The authoring environment did not execute the genuine-checkpoint Kaggle run. This card's populated results come from the run that creates it.

## Citations and attribution

- Harsh Bihany, Shubham Patel and Ashutosh Modi (2025). [LoRMA: Low-Rank Multiplicative Adaptation for LLMs](https://aclanthology.org/2025.findings-acl.527/). Identity-inflated multiplicative-adapter formulation.
- Chin-Yew Lin (2004). [ROUGE: A Package for Automatic Evaluation of Summaries](https://aclanthology.org/W04-1013/). Overlap evaluation; LCS recall also informs the custom alignment heuristic.
- Mohamed Elaraby, Yang Zhong and Diane Litman (2023). [Towards Argument-Aware Abstractive Summarization of Long Legal Opinions with Summary Reranking](https://aclanthology.org/2023.findings-acl.481/). Motivation for argument-role coverage; our reducer is not their reproduced system.
- Yusen Zhang et al. (2022). [SummN: A Multi-Stage Summarization Framework for Long Input Dialogues and Documents](https://aclanthology.org/2022.acl-long.112/). Context for split-then-summarize pipelines; our deterministic reduction differs from its learned stages.
- [Hugging Face Hub upload documentation](https://huggingface.co/docs/huggingface_hub/guides/upload) for artifact publication.

BibTeX entries are supplied in `CITATIONS.bib`. Base architecture/tokenizer attribution and lineage refer to the supplied Phase 2 v4 notebook; preserving that code does not establish a new pretrained-model license.
'''

def finalize(c):
    runtime=Path(c['runtime']);out=Path(c['submission_dir']);data=load_pickle(runtime/'prepared.pkl');rows=data['inputs']
    tok=PureByteLevelBPETokenizer.load(c['tokenizer_dir']);trained=read_shards(runtime,'lorma')
    chunk_map(rows,trained)
    tune_rows,tune_shards=subset_shards(rows,trained,'length_tune');tuning={}
    for length in c['summary_length_candidates']:
        predictions=assemble_predictions(tok,tune_rows,tune_shards,length);tuning[str(length)]=scores(tune_rows,predictions)
    selected=max(c['summary_length_candidates'],key=lambda length:(tuning[str(length)]['rougeL'],-length))
    report_rows,report_shards=subset_shards(rows,trained,'report')
    report_predictions=assemble_predictions(tok,report_rows,report_shards,selected)
    test_rows,test_shards=subset_shards(rows,trained,'test');predictions=assemble_predictions(tok,test_rows,test_shards,selected)
    test=[dict(id=p['id'],summary=p['summary']) for p in predictions]
    assert [p['id'] for p in test]==[r['id'] for r in test_rows] and len({p['id'] for p in test})==len(test_rows)
    assert all(isinstance(p['id'],str) and p['summary'].strip() for p in test)
    folder=out/'submission_lorma';folder.mkdir(exist_ok=True);atomic_json(test,folder/'prediction.json')
    with zipfile.ZipFile(out/'submission_lorma.zip','w',zipfile.ZIP_DEFLATED) as z:z.write(folder/'prediction.json','prediction.json')
    with zipfile.ZipFile(out/'submission_lorma.zip') as z:assert z.namelist()==['prediction.json'] and json.loads(z.read('prediction.json'))==test
    metrics=dict(selected_summary_tokens=selected,length_tuning=tuning,report=dict(lorma=scores(report_rows,report_predictions)),selection_rule='adapter: lowest token-weighted validation CE; final summary cap: highest ROUGE-L on disjoint length-tuning documents',elapsed_minutes_before_upload=(time.time()-c['started_at'])/60)
    atomic_json(metrics,out/'metrics.json');print(json.dumps(metrics,indent=2),flush=True)
    audit=json.loads((out/'data_audit.json').read_text());training=json.loads((out/'training_metrics.json').read_text())
    assert training['successful_updates']>=c['min_updates']
    assert training['epochs_completed']<=c['epochs'] and training['successful_updates']<=training['expected_updates']
    (out/'README.md').write_text(readme(c,audit,training,metrics),encoding='utf8');shutil.copy2('CITATIONS.bib',out/'CITATIONS.bib')
    atomic_json(c,out/'run_config.json')  # CONFIG deliberately never contains a token.
    source=out/'source';source.mkdir(exist_ok=True)
    for name in ['llama_scratch.py','bpe_tokenizer.py','phase6_v4_common.py','phase6_v4_engine.py','phase6_v4_report.py','phase6_v4_upload.py']:shutil.copy2(name,source/name)
    expected=['README.md','CITATIONS.bib','run_config.json','data_audit.json','training_history.json','training_metrics.json','metrics.json','submission_lorma/prediction.json','submission_lorma.zip']+['source/'+n for n in ['llama_scratch.py','bpe_tokenizer.py','phase6_v4_common.py','phase6_v4_engine.py','phase6_v4_report.py','phase6_v4_upload.py']]
    manifest={name:dict(bytes=(out/name).stat().st_size,sha256=sha256(out/name)) for name in expected}
    atomic_json(dict(files=manifest,contains_model_weights=False),out/'manifest.json')
    assert not list(out.rglob('*.pt')),'Prediction-only output must not contain weights'
    resolved=runtime.resolve();expected_runtime=Path(tempfile.gettempdir()).resolve()/'phase6_v4_lorma_runtime'
    assert resolved==expected_runtime and resolved.parent==Path(tempfile.gettempdir()).resolve(),'Refusing unsafe cleanup'
    assert not Path(c['checkpoint']).resolve().is_relative_to(resolved)
    shutil.rmtree(resolved);print('Complete predictions, README, citations and metrics saved. Temporary data removed; no model weights were written.',flush=True)

if __name__=='__main__':finalize(read_config())


HF credentials and write preflight, explicit artifact allowlist, retry handling and commit-pinned byte verification.

In [ ]:
%%writefile phase6_v4_upload.py
# Prediction-only Hugging Face upload, executed inside the notebook's bounded subprocess runner.
import os,json,time,hashlib,tempfile
from pathlib import Path

PREDICTION_FILES=['submission_lorma/prediction.json','submission_lorma.zip']

def config():return json.loads(Path('phase6_v4_config.json').read_text())

def credentials():
    token=os.environ.get('HF_WRITE_TOKEN')
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token=UserSecretsClient().get_secret('HF_WRITE')
        except Exception:token=os.environ.get('HF_TOKEN')
    if not token:raise RuntimeError('Enable Kaggle secret HF_WRITE with a Hugging Face token that can write to the configured repo, or set HF_WRITE_TOKEN.')
    return token

def retry(action,tries=3):
    for attempt in range(tries):
        try:return action()
        except Exception as error:
            status=getattr(getattr(error,'response',None),'status_code',None) or getattr(error,'code',None)
            if status in (400,401,403,404) or attempt+1==tries:raise
            time.sleep(2**(attempt+1))

def preflight(c):
    from huggingface_hub import HfApi
    api=HfApi(token=credentials());retry(lambda:api.whoami())
    retry(lambda:api.create_repo(c['hf_repo_id'],repo_type='model',private=True,exist_ok=True))
    # Older Hub clients have auth_check() but no write= parameter.
    import inspect
    checker=getattr(api,'auth_check',None)
    if checker is not None and 'write' in inspect.signature(checker).parameters:
        retry(lambda:checker(c['hf_repo_id'],repo_type='model',write=True))
    else:
        # Same official /auth-check/write endpoint used by current HfApi clients.
        from urllib.request import Request,urlopen
        url=getattr(api,'endpoint','https://huggingface.co').rstrip('/')+'/api/models/'+c['hf_repo_id']+'/auth-check/write'
        def check_write():
            request=Request(url,headers={'Authorization':'Bearer '+credentials()})
            with urlopen(request,timeout=20) as response:
                assert response.status==200,'Repository write access not confirmed'
        retry(check_write)
    print('HF write access verified for',c['hf_repo_id'],flush=True)
    return api

def upload(c):
    from huggingface_hub import hf_hub_download
    import zipfile
    root=Path(c['submission_dir']);prediction=root/PREDICTION_FILES[0];archive=root/PREDICTION_FILES[1]
    manifest=json.loads((root/'manifest.json').read_text())
    assert manifest['contains_model_weights'] is False
    expected=list(manifest['files'])+['manifest.json']
    assert set(PREDICTION_FILES+['README.md','CITATIONS.bib','metrics.json','training_metrics.json']).issubset(expected)
    assert all(not Path(name).is_absolute() and '..' not in Path(name).parts for name in expected)
    assert not any(Path(name).suffix in ['.pt','.pth','.pkl','.safetensors'] for name in expected)
    for name,entry in manifest['files'].items():
        assert hashlib.sha256((root/name).read_bytes()).hexdigest()==entry['sha256'],f'Local manifest mismatch: {name}'
    assert all((root/name).is_file() for name in expected),'Missing upload artifact'
    records=json.loads(prediction.read_text(encoding='utf8'))
    assert isinstance(records,list) and records and all(isinstance(r.get('id'),str) and isinstance(r.get('summary'),str) and r['summary'].strip() for r in records)
    assert len({r['id'] for r in records})==len(records)
    with zipfile.ZipFile(archive) as z:
        assert z.namelist()==['prediction.json']
        assert json.loads(z.read('prediction.json'))==records
    api=preflight(c)
    commit=retry(lambda:api.upload_folder(repo_id=c['hf_repo_id'],repo_type='model',folder_path=str(root),path_in_repo=c['hf_path_in_repo'],allow_patterns=expected,commit_message='Phase 6 v4: one-hour LoRMA32 predictions, model card, citations and metrics'))
    oid=commit.oid;token=credentials()
    # Read back at this exact commit; a successful API return alone is not verification.
    with tempfile.TemporaryDirectory(prefix='phase6_v4_hf_verify_') as cache:
        for name in expected:
            prefix=c['hf_path_in_repo'].strip('/')
            remote=(prefix+'/' if prefix else '')+name
            downloaded=retry(lambda remote=remote:hf_hub_download(repo_id=c['hf_repo_id'],filename=remote,repo_type='model',revision=oid,token=token,local_dir=cache,force_download=True))
            assert hashlib.sha256(Path(downloaded).read_bytes()).digest()==hashlib.sha256((root/name).read_bytes()).digest(),f'Upload hash mismatch: {remote}'
    print('HF upload verified byte-for-byte at commit',oid,flush=True)
    print('https://huggingface.co/'+c['hf_repo_id']+'/tree/'+oid+'/'+c['hf_path_in_repo'].strip('/'),flush=True)

if __name__=='__main__':
    import sys
    c=config()
    {'preflight':preflight,'upload':upload}[sys.argv[1]](c)


Method and evaluation references. These citations attribute LoRMA and ROUGE and identify research inspiration; custom heuristics are not claimed as reproductions.

In [ ]:
%%writefile CITATIONS.bib
@inproceedings{bihany-etal-2025-lorma,
  title = {{LoRMA}: Low-Rank Multiplicative Adaptation for {LLMs}},
  author = {Bihany, Harsh and Patel, Shubham and Modi, Ashutosh},
  booktitle = {Findings of the Association for Computational Linguistics: ACL 2025},
  year = {2025},
  url = {https://aclanthology.org/2025.findings-acl.527/}
}

@inproceedings{lin-2004-rouge,
  title = {{ROUGE}: A Package for Automatic Evaluation of Summaries},
  author = {Lin, Chin-Yew},
  booktitle = {Text Summarization Branches Out},
  year = {2004},
  pages = {74--81},
  url = {https://aclanthology.org/W04-1013/}
}

@inproceedings{elaraby-etal-2023-towards,
  title = {Towards Argument-Aware Abstractive Summarization of Long Legal Opinions with Summary Reranking},
  author = {Elaraby, Mohamed and Zhong, Yang and Litman, Diane},
  booktitle = {Findings of the Association for Computational Linguistics: ACL 2023},
  year = {2023},
  url = {https://aclanthology.org/2023.findings-acl.481/}
}

@inproceedings{zhang-etal-2022-summ,
  title = {{SummN}: A Multi-Stage Summarization Framework for Long Input Dialogues and Documents},
  author = {Zhang, Yusen and Ni, Ansong and Mao, Ziming and Wu, Chen Henry and Zhu, Chenguang and Deb, Budhaditya and Awadallah, Ahmed and Radev, Dragomir and Zhang, Rui},
  booktitle = {Proceedings of the 60th Annual Meeting of the Association for Computational Linguistics (Volume 1: Long Papers)},
  year = {2022},
  url = {https://aclanthology.org/2022.acl-long.112/}
}


In [ ]:
# Check Hugging Face write access before preparation and training
run_checked([sys.executable,'-u','phase6_v4_upload.py','preflight'])


In [ ]:
# Prepare sentence-aware windows, improved targets and disjoint holdouts
run_checked([sys.executable,'-u','phase6_v4_engine.py','prepare'])


In [ ]:
# Train ONE LoRMA32 model within the time budget and generate on BOTH GPUs
run_checked([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nproc_per_node=2','phase6_v4_engine.py','distributed'])


In [ ]:
# Select summary length, report quality, package predictions and write the README
run_checked([sys.executable,'-u','phase6_v4_report.py'])
from IPython.display import display,FileLink,Markdown
out=Path(CONFIG['submission_dir'])
training=json.loads((out/'training_metrics.json').read_text())
assert training['successful_updates']>=CONFIG['min_updates']
assert training['successful_updates']<=training['expected_updates']
print('Actual completed epoch fraction:',training['fractional_epochs_completed'])
assert not list(out.rglob('*.pt'))
assert not Path(CONFIG['runtime']).exists(),'Temporary runtime was not cleaned.'
print(json.dumps(json.loads((out/'metrics.json').read_text()),indent=2))
display(Markdown((out/'README.md').read_text(encoding='utf8')))
for name in ['submission_lorma/prediction.json','submission_lorma.zip','README.md','CITATIONS.bib','metrics.json','training_metrics.json','data_audit.json','manifest.json']:
    display(FileLink(str(out/name)))


In [ ]:
# Upload the prediction artifacts, README, citations and metrics; verify remote bytes
run_checked([sys.executable,'-u','phase6_v4_upload.py','upload'])
print(f'One-model run and verified HF upload finished in {(time.time()-RUN_STARTED_AT)/60:.2f} minutes.')
print('Repository: https://huggingface.co/'+CONFIG['hf_repo_id'])
assert time.time()-RUN_STARTED_AT<60*60,'Artifacts exist, but this execution exceeded the one-hour requirement.'


## Interpretation and limitations
Use the actual `fractional_epochs_completed`, `stopped_for_time`, successful updates and best validation CE in `training_metrics.json`. Two epochs are a maximum training target, not a promise within one hour. A timeout cannot be interpreted as a complete submission.

The 64-document generation report is disjoint from training, loss-checkpoint selection and the 32-document length-tuning group. Report ROUGE and output lengths are validation diagnostics, not official test scores or proof of improvement over an unrun baseline. No ablation study is performed.

Initial overlapping windows cover every source token; inference deliberately selects a measured majority (at least 85%) rather than every window. Alignment, sentence segmentation, role labels, source-support scoring and reduction are heuristic and do not guarantee factuality or retention of every legal fact. The reducer is deterministic, not an untrained model rewrite or a reproduction of the cited research systems.

CPU mechanics, v4-specific attention, LoRMA updates/merge, coverage/alignment, schedule boundaries, partial-epoch reporting, prediction packaging, README/manifest generation and mocked HF upload verification were checked. Actual genuine-checkpoint T4 ×2/NCCL execution, score gains, live token permissions and wall-clock runtime require the real run. No model weights are saved, so interrupted training cannot resume.
